# Flood Segmentation — YOLOv8s-seg on FloodNet 2021

Self-contained Colab notebook that:
1. Downloads **FloodNet 2021** (Hurricane Harvey UAV imagery, ~2343 images, 10 semantic classes)
2. Converts pixel masks to YOLO polygon labels
3. Trains **YOLOv8s-seg** for 50 epochs on a free T4
4. Evaluates mAP / mask metrics on the test split
5. Runs inference on user-provided flood photos
6. Packages `flood.pt` for download

**Runtime:** Free T4 GPU (Runtime → Change runtime type → T4 GPU).
**Time:** ~75–110 min end-to-end.

Dataset source: [aletbm/aerial-imagery-dataset-floodnet-challenge (Kaggle)](https://www.kaggle.com/datasets/aletbm/aerial-imagery-dataset-floodnet-challenge) — mirror of the [FloodNet 2021 challenge](https://github.com/BinaLab/FloodNet-Supervised_v1.0).

Classes after conversion (kept identical to the source, 10 total):
`background, building_flooded, building_non-flooded, road_flooded, road_non-flooded, water, tree, vehicle, pool, grass`.

## 0. GPU check + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.3.40 kagglehub Pillow tqdm opencv-python-headless pyyaml

## 1. Download FloodNet 2021

Uses kagglehub. If you've never used kagglehub on this Colab account it'll ask you to upload a `kaggle.json` token — get it from https://www.kaggle.com/settings/account → Create New Token.

In [ ]:
import kagglehub
from pathlib import Path

raw = Path(kagglehub.dataset_download('aletbm/aerial-imagery-dataset-floodnet-challenge'))
print('FloodNet downloaded to:', raw)
for p in sorted(raw.rglob('*'))[:60]:
    print(' ', p.relative_to(raw))

## 2. Convert RGB masks → YOLO segmentation polygons

FloodNet ships per-pixel masks where each class is encoded as a grayscale ID (0..9). We:
1. Walk every image / mask pair
2. Extract polygon contours per class
3. Normalise coordinates to `[0,1]`
4. Write Ultralytics-format `labels/*.txt`
5. Generate `data.yaml`

In [ ]:
import cv2, shutil, yaml
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Canonical FloodNet class ids → human-readable names.
# Order matters — index = class id used by the YOLO labels.
NAMES = [
    'background',           # 0
    'building_flooded',     # 1
    'building_non-flooded', # 2
    'road_flooded',         # 3
    'road_non-flooded',     # 4
    'water',                # 5
    'tree',                 # 6
    'vehicle',              # 7
    'pool',                 # 8
    'grass',                # 9
]

# Skip background polygons — they're noise for YOLO.
SKIP_CLASS_IDS = {0}

DST = Path('/content/floodnet_yolo')
for sub in ['images/train', 'images/val', 'images/test', 'labels/train', 'labels/val', 'labels/test']:
    (DST / sub).mkdir(parents=True, exist_ok=True)

# Auto-discover image+mask pairs. FloodNet's layout has split dirs like:
#   .../train/train-org-img/*.jpg  + .../train/train-label-img/*.png
# Newer mirrors may use a different layout — handle both by globbing.
ALL = []
for img_path in raw.rglob('*'):
    if img_path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
        continue
    parts_lower = [p.lower() for p in img_path.parts]
    if any('label' in p or 'mask' in p for p in parts_lower):
        continue  # this IS a mask, skip
    # try to find its mask sibling
    candidates = []
    name = img_path.stem
    for cand_root in img_path.parents:
        for label_dir in ['label', 'labels', 'mask', 'masks', 'train-label-img', 'val-label-img', 'test-label-img']:
            for ext in ['.png', '.jpg']:
                candidates.append(cand_root / label_dir / f'{name}_lab{ext}')
                candidates.append(cand_root / label_dir / f'{name}{ext}')
    mask_path = next((c for c in candidates if c.exists()), None)
    if mask_path:
        # split inference from path component
        if 'val' in parts_lower or 'valid' in parts_lower:
            split = 'val'
        elif 'test' in parts_lower:
            split = 'test'
        else:
            split = 'train'
        ALL.append((img_path, mask_path, split))

print(f'Found {len(ALL)} image/mask pairs')
assert len(ALL) > 500, 'Too few pairs — inspect the dataset folder above and adjust the glob.'

In [ ]:
def mask_to_yolo_lines(mask_gray: np.ndarray, w: int, h: int) -> list[str]:
    """Convert a single-channel class-id mask into YOLO polygon label lines."""
    lines = []
    for cls_id in np.unique(mask_gray):
        cls_id = int(cls_id)
        if cls_id in SKIP_CLASS_IDS or cls_id >= len(NAMES):
            continue
        binary = (mask_gray == cls_id).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            if cv2.contourArea(c) < 50:  # filter very small specks
                continue
            poly = c.squeeze()
            if poly.ndim != 2 or len(poly) < 3:
                continue
            norm = poly.astype(float)
            norm[:, 0] /= w
            norm[:, 1] /= h
            coords = ' '.join(f'{x:.6f} {y:.6f}' for x, y in norm)
            lines.append(f'{cls_id} {coords}')
    return lines

split_counts = {'train': 0, 'val': 0, 'test': 0}
for img_path, mask_path, split in tqdm(ALL, desc='Converting'):
    img = cv2.imread(str(img_path))
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if img is None or mask is None:
        continue
    if mask.ndim == 3:
        # if shipped as RGB, take the first non-zero channel; FloodNet usually ships single-channel
        mask = mask[..., 0]
    h, w = img.shape[:2]
    lines = mask_to_yolo_lines(mask, w, h)
    if not lines:
        continue
    new_name = img_path.stem + img_path.suffix
    shutil.copy(img_path, DST / 'images' / split / new_name)
    (DST / 'labels' / split / (img_path.stem + '.txt')).write_text('\n'.join(lines))
    split_counts[split] += 1

print('Converted:', split_counts)
assert split_counts['train'] > 0, 'No training samples produced.'

# If the source had no val/test split, carve one out of train
if split_counts['val'] == 0 or split_counts['test'] == 0:
    import random
    random.seed(42)
    train_imgs = list((DST / 'images/train').glob('*'))
    random.shuffle(train_imgs)
    n = len(train_imgs)
    val_take, test_take = train_imgs[: n // 10], train_imgs[n // 10 : 2 * n // 10]
    for target_split, take in [('val', val_take), ('test', test_take)]:
        if split_counts[target_split] > 0:
            continue
        for img in take:
            shutil.move(img, DST / 'images' / target_split / img.name)
            lbl = DST / 'labels/train' / (img.stem + '.txt')
            if lbl.exists():
                shutil.move(lbl, DST / 'labels' / target_split / lbl.name)
        split_counts[target_split] = len(take)
        split_counts['train'] -= len(take)
    print('After carving:', split_counts)

(DST / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(DST),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    len(NAMES),
    'names': NAMES,
}))
print((DST / 'data.yaml').read_text())

### Sanity check: visualise a labelled training sample

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random

PALETTE = [(0,0,0), (255,0,0), (255,140,0), (30,144,255), (135,206,250),
           (0,191,255), (34,139,34), (255,255,0), (0,255,255), (124,252,0)]

samples = random.sample(list((DST / 'images/train').glob('*')), k=4)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_path in zip(axes.flat, samples):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    overlay = img.copy()
    draw = ImageDraw.Draw(overlay, 'RGBA')
    lbl = DST / 'labels/train' / (img_path.stem + '.txt')
    for line in lbl.read_text().splitlines():
        toks = line.split()
        cls = int(toks[0])
        pts = [(float(toks[i]) * W, float(toks[i+1]) * H) for i in range(1, len(toks), 2)]
        rgb = PALETTE[cls % len(PALETTE)]
        draw.polygon(pts, fill=rgb + (90,), outline=rgb + (255,))
    ax.imshow(overlay); ax.set_title(img_path.name, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Train YOLOv8s-seg

Segmentation is heavier than detection so we keep batch=16. Expect ~60–80 min on T4.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-seg.pt')

results = model.train(
    data=str(DST / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    name='flood_yolov8s_seg',
    device=0,
    workers=2,
    seed=42,
    verbose=True,
)

## 4. Evaluate on test split

In [ ]:
from pathlib import Path
best = Path('runs/segment/flood_yolov8s_seg/weights/best.pt')
model = YOLO(str(best))
metrics = model.val(data=str(DST / 'data.yaml'), split='test', imgsz=640, plots=True)
print('Box mAP@0.5      :', round(float(metrics.box.map50), 4))
print('Mask mAP@0.5     :', round(float(metrics.seg.map50), 4))
print('Mask mAP@0.5:0.95:', round(float(metrics.seg.map), 4))
print('Mask Precision   :', round(float(metrics.seg.mp), 4))
print('Mask Recall      :', round(float(metrics.seg.mr), 4))

## 5. Sanity-check on real flood photos

In [ ]:
from pathlib import Path
TEST_DIR = Path('/content/test_flood')
TEST_DIR.mkdir(exist_ok=True)

# A few open-licensed flood photos for a quick visual check.
samples = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/0/04/2010_Pakistan_floods_2.jpg/1024px-2010_Pakistan_floods_2.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/0/0c/Houston_I-10_at_East_Tx_Loop_-_FEMA_Aerial.jpg/1024px-Houston_I-10_at_East_Tx_Loop_-_FEMA_Aerial.jpg',
]
for i, url in enumerate(samples):
    out = TEST_DIR / f'sample_{i}.jpg'
    if not out.exists():
        !wget -q -O {out} {url}

results = model.predict(source=str(TEST_DIR), imgsz=640, conf=0.25, save=True)
from IPython.display import Image as IPyImage, display
import glob
for p in sorted(glob.glob(str(Path(results[0].save_dir) / '*.jpg'))):
    print(p); display(IPyImage(p))

## 6. Package for local use

In [ ]:
import shutil
from google.colab import files

src = Path('runs/segment/flood_yolov8s_seg')
out = Path('/content/flood_model_release'); out.mkdir(exist_ok=True)
shutil.copy(src / 'weights/best.pt', out / 'flood.pt')
for f in ['results.png', 'results.csv', 'confusion_matrix.png', 'PR_curve.png', 'MaskPR_curve.png']:
    if (src / f).exists():
        shutil.copy(src / f, out / f)

shutil.make_archive('/content/flood_model_release', 'zip', out)
files.download('/content/flood_model_release.zip')

## Drop the trained weights into the app

Unzip `flood_model_release.zip` and copy `flood.pt` to `model/flood.pt`. Then set:

```bash
APP_FLOOD_SEGMENTER_WEIGHTS=$(pwd)/model/flood.pt uvicorn app.main:app --reload
```

(or just rename it over the existing `runs/segment/flood_seg/weights/best.pt` if you prefer to keep the default path).

The flood service in [`app/services/flood.py`](../app/services/flood.py) tolerates the new 10-class FloodNet vocabulary — it matches `flood/water/flooded` for the inundated area and groups buildings/vehicles/plants by class-name pattern.